In [0]:
%pip install -U mlflow transformers torch huggingface_hub sentencepiece accelerate pandas

In [0]:
dbutils.library.restartPython()

In [0]:
  import re
  import unicodedata
  import pandas as pd
  import torch
  import torch.nn as nn
  import mlflow.pyfunc

  from huggingface_hub import hf_hub_download
  from transformers import AutoTokenizer, AutoModel

  BASE_MODEL = "UBC-NLP/MARBERTv2"
  REPO_ID = "amitca71/marbertv2-levantine-incitement-detector"
  MAX_LENGTH = 160
  USE_NORMALIZED_TEXT_FOR_MODEL = False
  POOLING_STRATEGY = "cls"

  ARABIC_DIACRITICS = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]')

  def normalize_arabic(text: str) -> str:
      text = unicodedata.normalize("NFKC", text or "").strip()
      text = ARABIC_DIACRITICS.sub("", text)
      text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
      text = text.replace("ى", "ي").replace("ؤ", "و").replace("ئ", "ي").replace("ة", "ه")
      text = re.sub(r"[^\w\s#@/]", " ", text)
      text = re.sub(r"\s+", " ", text)
      return text.strip().lower()

  def text_for_model(text: str, use_normalized: bool = USE_NORMALIZED_TEXT_FOR_MODEL) -> str:
      return normalize_arabic(text) if use_normalized else (text or "").strip()

  class MarbertMultiTask(nn.Module):
      def __init__(self, base_model_path: str, pooling_strategy: str = "cls"):
          super().__init__()
          self.encoder = AutoModel.from_pretrained(base_model_path)
          self.pooling_strategy = pooling_strategy
          hidden_size = self.encoder.config.hidden_size

          if pooling_strategy in {"cls", "mean", "max"}:
              rep_dim = hidden_size
          elif pooling_strategy == "cls_mean_max":
              rep_dim = hidden_size * 3
          else:
              raise ValueError(f"Unknown pooling_strategy: {pooling_strategy}")

          self.classifier = nn.Linear(rep_dim, 3)
          self.lexicon_head = nn.Linear(rep_dim, 1)

      def masked_mean_pool(self, last_hidden_state, attention_mask):
          mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
          masked_embeddings = last_hidden_state * mask
          summed = masked_embeddings.sum(dim=1)
          counts = mask.sum(dim=1).clamp(min=1e-9)
          return summed / counts

      def masked_max_pool(self, last_hidden_state, attention_mask):
          mask = attention_mask.unsqueeze(-1).bool()
          masked = last_hidden_state.masked_fill(~mask, float("-inf"))
          pooled = masked.max(dim=1).values
          pooled[torch.isinf(pooled)] = 0.0
          return pooled

      def forward(self, input_ids, attention_mask, token_type_ids=None):
          outputs = self.encoder(
              input_ids=input_ids,
              attention_mask=attention_mask,
              token_type_ids=token_type_ids,
              return_dict=True,
          )

          last_hidden_state = outputs.last_hidden_state
          cls_vec = last_hidden_state[:, 0, :]
          mean_vec = self.masked_mean_pool(last_hidden_state, attention_mask)
          max_vec = self.masked_max_pool(last_hidden_state, attention_mask)

          if self.pooling_strategy == "cls":
              pooled = cls_vec
          elif self.pooling_strategy == "mean":
              pooled = mean_vec
          elif self.pooling_strategy == "max":
              pooled = max_vec
          elif self.pooling_strategy == "cls_mean_max":
              pooled = torch.cat([cls_vec, mean_vec, max_vec], dim=-1)
          else:
              raise ValueError(f"Unknown pooling_strategy: {self.pooling_strategy}")

          return {
              "logits": self.classifier(pooled),
              "lexicon_logits": self.lexicon_head(pooled),
          }

  class IncitementDetectorModel(mlflow.pyfunc.PythonModel):
      def load_context(self, context):
          self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

          tokenizer_path = context.artifacts["tokenizer"]
          base_model_path = context.artifacts["base_model"]
          checkpoint_path = context.artifacts["checkpoint"]

          self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
          self.model = MarbertMultiTask(base_model_path, pooling_strategy=POOLING_STRATEGY)

          state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
          self.model.load_state_dict(state_dict)
          self.model.to(self.device)
          self.model.eval()

          self.label_map = {0: "normal", 1: "abusive", 2: "incitement"}

      def _predict_one(self, text: str):
          encoded = self.tokenizer(
              text_for_model(text),
              truncation=True,
              padding="max_length",
              max_length=MAX_LENGTH,
              return_tensors="pt",
          )
          encoded = {k: v.to(self.device) for k, v in encoded.items()}

          with torch.no_grad():
              out = self.model(**encoded)

          logits = out["logits"]
          lexicon_logits = out["lexicon_logits"]

          probs = torch.softmax(logits, dim=-1).squeeze(0).tolist()
          pred_id = int(torch.argmax(logits, dim=-1).item())

          return {
              "pred_label": self.label_map[pred_id],
              "confidence": float(probs[pred_id]),
              "prob_normal": float(probs[0]),
              "prob_abusive": float(probs[1]),
              "prob_incitement": float(probs[2]),
              "lexicon_signal_prob": float(torch.sigmoid(lexicon_logits).squeeze(0).item()),
          }

      def predict(self, context, model_input):
          if isinstance(model_input, pd.DataFrame):
              texts = model_input["text"].fillna("").astype(str).tolist()
          elif isinstance(model_input, list):
              texts = [str(x) for x in model_input]
          else:
              raise ValueError("Input must be a pandas DataFrame with a `text` column.")

          rows = [self._predict_one(text) for text in texts]
          return pd.DataFrame(rows)

In [0]:
checkpoint_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="model.pt",
)

In [0]:
catalog = "amit"
schema = "default"
registered_model_name = f"{catalog}.{schema}.marbertv2_levantine_incitement_detector"

input_example = pd.DataFrame({
    "text": ["انت يا عميل السفارات يا ابن الكلب حسابك عسير"]
})

In [0]:
  import os
  import shutil
  import tempfile
  from transformers import AutoTokenizer, AutoModel
  from huggingface_hub import hf_hub_download

  work_dir = tempfile.mkdtemp(prefix="marbert_bundle_")

  base_model_dir = os.path.join(work_dir, "base_model")
  tokenizer_dir = os.path.join(work_dir, "tokenizer")
  os.makedirs(base_model_dir, exist_ok=True)
  os.makedirs(tokenizer_dir, exist_ok=True)

  tok = AutoTokenizer.from_pretrained(BASE_MODEL)
  tok.save_pretrained(tokenizer_dir)

  base = AutoModel.from_pretrained(BASE_MODEL)
  base.save_pretrained(base_model_dir)

  src_checkpoint = hf_hub_download(repo_id=REPO_ID, filename="model.pt")
  checkpoint_path = os.path.join(work_dir, "model.pt")
  shutil.copyfile(src_checkpoint, checkpoint_path)



In [0]:
  with mlflow.start_run():
      model_info = mlflow.pyfunc.log_model(
          artifact_path="model",
          python_model=IncitementDetectorModel(),
          artifacts={
              "checkpoint": checkpoint_path,
              "base_model": base_model_dir,
              "tokenizer": tokenizer_dir,
          },
          registered_model_name=registered_model_name,
          input_example=input_example,
          pip_requirements=[
              "mlflow",
              "torch",
              "transformers",
              "huggingface_hub",
              "sentencepiece",
              "accelerate",
              "pandas",
          ],
      )

In [0]:
  loaded = mlflow.pyfunc.load_model(model_info.model_uri)
  loaded.predict(pd.DataFrame({"text": ["انت يا عميل السفارات يا ابن الكلب حسابك عسير"]}))

In [0]:
  import os
  import requests
  DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

  workspace_url = "https://dbc-de54b796-a6c4.cloud.databricks.com/"
  endpoint_name = "marbertv2-incitement-detector"
  token = DATABRICKS_TOKEN

  payload = {
      "dataframe_records": [
          {"text": "انت يا عميل السفارات يا ابن الكلب حسابك عسير"}
      ]
  }

  resp = requests.post(
      f"{workspace_url}/serving-endpoints/{endpoint_name}/invocations",
      headers={
          "Authorization": f"Bearer {token}",
          "Content-Type": "application/json",
      },
      json=payload,
      timeout=120,
  )

  print(resp.status_code)
  print(resp.json())

In [0]:
loaded.predict(pd.DataFrame({"text": ["انت يا عميل السفارات يا ابن الكلب حسابك عسير"]}))